[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_09/TSA_ch9_neural_networks/TSA_ch9_neural_networks.ipynb)

# TSA_ch9_neural_networks

A multilayer perceptron and its activation functions; a recurrent network unrolled in time and the LSTM cell; a small multi-output MLP and a small LSTM (PyTorch, CPU, fixed seed) giving 14-day forecasts of the daily load of Romania from one forecast origin.

*Time Series Analysis (TSA), Chapter 9: Machine learning for time series. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_9'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'   # small data: one thread per model fit is faster
# PyTorch is preinstalled in Google Colab; it is needed only for the LSTM

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # default size about 7 inches wide (as in MFM); a chart is shown 11-13 cm wide on a slide, so these font sizes
    # give 7-9 pt text there. Multi-panel charts should stay near 7-7.6 inches wide: at 10-11 inches the axis
    # text drops to about 5 pt on the slide.
    rc['figure.figsize'] = (7.0, 3.4)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)
    _remember_tight_layout()


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import math

from matplotlib.patches import Rectangle, Circle, FancyArrowPatch

from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor)

from sklearn.inspection import permutation_importance

from sklearn.linear_model import LassoCV, LinearRegression, LogisticRegression, RidgeCV, lasso_path

from sklearn.metrics import roc_auc_score

from sklearn.model_selection import KFold, TimeSeriesSplit

from sklearn.neural_network import MLPRegressor

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeRegressor, plot_tree

from dateutil.easter import easter, EASTER_ORTHODOX

HERE = "."

SEED = 2026
# daily load: the file and the evaluation design of Chapter 4 (26 origins, every 14 days, horizon 14 days)
LOAD_FILE = 'ch4_ro_load_hourly.csv'
LOAD_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/Quantlets/Ch_04/'
LOAD_END = '2026-06-30'
CV_FIRST, CV_STEP, CV_H = '2025-06-30', 14, 14
LAGS = 14                      # y_t, ..., y_{t-13} at the forecast origin
HICP = ('prc_hicp_minr', 'M.I15.TOTAL.{}')
EU27 = ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV',
        'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']
INF_START, INF_OOS = '2005-01-01', 2016          # inflation targeting in Romania since August 2005; tests from 2016
INF_H = [1, 3, 6, 12]
RV_H = 5                                          # volatility target: mean variance proxy over the next 5 days
RV_OOS = {'sp500': '2013-01-01', 'dax': '2013-01-01'}
OHLC_START = {'sp500': '2008-01-01', 'dax': '2008-01-01'}
HAR = ['d', 'w', 'm']
EXT = HAR + ['d1', 'd2', 'd3', 'd4', 'q', 'r', 'r_neg', 'abs_r']
SIGN_FEATS = ['r0', 'r1', 'r2', 'r3', 'r4', 'm5', 'm21', 'm63', 'v21', 'v63']
NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET'}
MCOL = {'Seasonal naive': st.Amber, 'ETS': st.Teal, 'SARIMA': st.Purple, 'DHR': st.MainBlue, 'Combination': st.Crimson,
        'Ridge': st.Teal, 'RF': st.Forest, 'GB direct': st.IDAred, 'GB recursive': st.Orange, 'MLP': '#6B8E23',
        'LSTM': st.Purple, 'RW': st.Amber, 'AR': st.MainBlue, 'Lasso': st.Teal, 'GB local': st.IDAred,
        'GB global': st.Orange, 'HAR': st.MainBlue, 'GB': st.IDAred, 'Mean': st.Amber, 'Logit': st.MainBlue}
GB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0)

FEATS = ['lag0', 'lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'lag6', 'lag7', 'lag8', 'lag9', 'lag10', 'lag11', 'lag12', 'lag13', 'mean7', 'mean28', 'same_dow', 'mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sin1', 'cos1', 'sin2', 'cos2', 'easter', 'xmas', 'holiday']

INF_FEATS = ['pi0', 'pi1', 'pi2', 'pi3', 'pi6', 'pi12', 'm0', 'm1', 'm2', 'm3s', 'm6s', 'month']

M4_FILE = 'ch9_m4_owa.csv'

HIGHLIGHT = {'lag0': st.MainBlue, 'same_dow': st.IDAred, 'mean7': st.Forest, 'sat': st.Orange, 'holiday': st.Purple, 'easter': st.Amber, 'xmas': st.Teal}

M4_TYPES = {'Statistical': st.MainBlue, 'Combination (S)': st.Teal, 'Combination (S & ML)': st.Purple, 'Combination (ML)': st.Orange, 'Machine Learning': st.IDAred, 'Hybrid': st.Forest, 'Other': st.Amber}

GKX_MODELS = ['OLS-3+H', 'PLS', 'PCR', 'ENet+H', 'GLM+H', 'RF', 'GBRT+H', 'NN1', 'NN2', 'NN3', 'NN4', 'NN5']

GKX_R2 = [0.16, 0.27, 0.26, 0.11, 0.19, 0.33, 0.34, 0.33, 0.39, 0.4, 0.39, 0.36]

GKX_SR = [0.61, 0.72, 0.88, 0.39, 0.76, 0.98, 0.81, 1.17, 1.16, 1.2, 1.35, 1.15]

def ro_holidays(years):
    """Romanian public holidays (Orthodox Easter and Pentecost from dateutil): date -> name (as in Chapter 4)."""
    out = {}
    for y in years:
        e = pd.Timestamp(easter(y, EASTER_ORTHODOX))
        day = pd.Timedelta(days=1)
        hol = {'New Year': [f'{y}-01-01', f'{y}-01-02'], 'Union Day': [f'{y}-01-24'],
               'Easter': [e - 2 * day, e, e + day], 'Labour Day': [f'{y}-05-01'], "Children's Day": [f'{y}-06-01'],
               'Pentecost': [e + 49 * day, e + 50 * day], 'Assumption': [f'{y}-08-15'], "St Andrew's Day": [f'{y}-11-30'],
               'National Day': [f'{y}-12-01'], 'Christmas': [f'{y}-12-25', f'{y}-12-26']}
        if y >= 2024:
            hol['Epiphany'] = [f'{y}-01-06', f'{y}-01-07']
        for k, v in hol.items():
            for x in v:
                out[pd.Timestamp(x)] = k
    return pd.Series(out, name='holiday').sort_index()


def load_daily(end=LOAD_END):
    """Daily mean electricity load of Romania in GW (ENTSO-E hourly values, Chapter 4 extract, Romanian winter time;
    isolated one-hour glitches and missing hours interpolated), 1 January 2022 - 30 June 2026."""
    local = [os.path.join(HERE, '..', 'Ch_04', LOAD_FILE)] + [os.path.join(d, 'Quantlets', 'Ch_04', LOAD_FILE)
                                                              for d in ('.', '..', '../..', '../../..')]
    src = next((p for p in local if os.path.exists(p)), LOAD_RAW + LOAD_FILE)
    s = pd.read_csv(src, index_col=0, parse_dates=True).iloc[:, 0]
    s = s.loc[:pd.Timestamp(end) + pd.Timedelta(hours=21)]
    s.index = s.index + pd.Timedelta(hours=2)
    s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq='h'))
    r = s / ((s.shift(1) + s.shift(-1)) / 2)
    s[(r < 0.7) | (r > 1.3)] = np.nan
    d = s.interpolate().resample('D').mean() / 1000
    return d.loc['2022-01-01':end].rename('load')


def calendar(idx):
    """Calendar features of the dates idx: weekday dummies (Sunday is the base), two annual Fourier pairs, Orthodox
    Easter (Friday to Monday), the Christmas - New Year days (24 December - 2 January) and other public holidays."""
    idx = pd.DatetimeIndex(idx)
    X = pd.DataFrame(index=idx)
    for j, nm in enumerate(['mon', 'tue', 'wed', 'thu', 'fri', 'sat']):
        X[nm] = (idx.dayofweek == j).astype(float)
    doy = idx.dayofyear.values
    for k in (1, 2):
        X[f'sin{k}'] = np.sin(2 * np.pi * k * doy / 365.25)
        X[f'cos{k}'] = np.cos(2 * np.pi * k * doy / 365.25)
    hol = ro_holidays(range(idx[0].year, idx[-1].year + 1)).reindex(idx)
    X['easter'] = (hol == 'Easter').astype(float).values
    X['xmas'] = (((idx.month == 12) & (idx.day >= 24)) | ((idx.month == 1) & (idx.day <= 2))).astype(float)
    X['holiday'] = (hol.notna().values & (X['easter'] == 0).values & (X['xmas'] == 0).values).astype(float)
    return X


def direct_frame(y, h):
    """Supervised table for horizon h: one row per forecast origin t. Features known at t: y_t, ..., y_{t-13}, the
    7- and 28-day means, the last value of the same weekday as the target (y_{t+h-7k}, k = ceil(h/7)) and the calendar
    of the target day t+h. Target: y_{t+h}."""
    X = pd.DataFrame({f'lag{j}': y.shift(j) for j in range(LAGS)}, index=y.index)
    X['mean7'] = y.rolling(7).mean()
    X['mean28'] = y.rolling(28).mean()
    k = math.ceil(h / 7)
    X['same_dow'] = y.shift(7 * k - h)
    cal = calendar(y.index + pd.Timedelta(days=h))
    cal.index = y.index
    X = pd.concat([X, cal], axis=1)
    X['y'] = y.shift(-h)
    return X.iloc[27:]


def origins(y=None, first=CV_FIRST, step=CV_STEP, h=CV_H):
    """The forecast origins of Chapter 4: every 14 days from 30 June 2025, while 14 days of data remain."""
    y = load_daily() if y is None else y
    return list(pd.date_range(first, y.index[-1] - pd.Timedelta(days=h), freq=f'{step}D'))


def oos_r2(y, yhat, ybar):
    """Out-of-sample R^2 against a benchmark forecast ybar: 1 - sum (y - yhat)^2 / sum (y - ybar)^2."""
    y, yhat, ybar = (np.asarray(a, float) for a in (y, yhat, ybar))
    return float(1 - np.sum((y - yhat) ** 2) / np.sum((y - ybar) ** 2))


def dm_raw(l1, l2, h=1):
    """Diebold-Mariano test on two loss series (Chapter 4): HLN small-sample correction, t(n - 1) p-value.
    d_t = l1_t - l2_t: a negative mean favours the first forecast."""
    d = np.asarray(l1, float) - np.asarray(l2, float)
    n = len(d)
    dbar = d.mean()
    g = [np.mean((d[k:] - dbar) * (d[:n - k] - dbar)) for k in range(h)]
    v = (g[0] + 2 * sum(g[1:])) / n
    dm = dbar / np.sqrt(v)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n) * dm
    return {'dbar': float(dbar), 'dm': float(dm), 'hln': float(hln), 'p': float(2 * stats.t.sf(abs(hln), n - 1)), 'n': int(n)}


def make_ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13), cv=TimeSeriesSplit(5)))


def make_rf(seed=SEED, n=200):
    return RandomForestRegressor(n_estimators=n, min_samples_leaf=3, max_features=0.5, n_jobs=-1, random_state=seed)


def make_gb(seed=SEED, **kw):
    return HistGradientBoostingRegressor(random_state=seed, **{**GB_PARAMS, **kw})


def direct_models():
    """The direct-strategy learners: ridge, random forest, gradient boosting."""
    return {'Ridge': make_ridge, 'RF': make_rf, 'GB direct': make_gb}


def forecast_direct(y, origin, make, h_max=CV_H, frames=None, feats=None):
    """Direct strategy: one model per horizon h = 1..h_max, trained on the origins t with t + h <= origin."""
    feats = feats or FEATS
    out = []
    for h in range(1, h_max + 1):
        F = frames[h] if frames else direct_frame(y, h)
        tr = F.loc[:origin - pd.Timedelta(days=h)].dropna()
        m = make().fit(tr[feats].values, tr['y'].values)
        out.append(float(m.predict(F.loc[[origin], feats].values)[0]))
    return np.array(out)


def forecast_recursive(y, origin, make, h_max=CV_H):
    """Recursive strategy: one one-step model; its forecast replaces the unknown value and the step is repeated."""
    F = direct_frame(y.loc[:origin], 1)
    tr = F.dropna()
    m = make().fit(tr[FEATS].values, tr['y'].values)
    ext = y.loc[:origin].copy()
    out = []
    for h in range(1, h_max + 1):
        row = direct_frame(ext.iloc[-40:], 1).iloc[[-1]]
        f = float(m.predict(row[FEATS].values)[0])
        out.append(f)
        ext.loc[ext.index[-1] + pd.Timedelta(days=1)] = f
    return np.array(out)


def mimo_inputs(y, t, window=28, h_max=CV_H):
    """Multi-output (MIMO) inputs at origin t: the last `window` values, the weekday of t (one-hot) and the
    holiday flags of the next h_max days."""
    past = y.loc[:t].values[-window:]
    dow = np.eye(7)[t.dayofweek]
    cal = calendar(pd.date_range(t + pd.Timedelta(days=1), periods=h_max, freq='D'))
    hol = (cal[['easter', 'xmas', 'holiday']].sum(axis=1) > 0).astype(float).values
    return np.concatenate([past, dow, hol])


class MLPMulti:
    """Average of three small MLPs with 14 outputs (one per horizon); inputs and targets standardised."""

    def __init__(self, hidden=(32,), alpha=1e-2, n_models=3, epochs=500, seed=SEED):
        self.hidden, self.alpha, self.n_models, self.epochs, self.seed = hidden, alpha, n_models, epochs, seed

    def fit(self, X, Y):
        self.sx = StandardScaler().fit(X)
        self.my, self.sy = Y.mean(), Y.std()
        Z, T = self.sx.transform(X), (Y - self.my) / self.sy
        self.models = [MLPRegressor(hidden_layer_sizes=self.hidden, alpha=self.alpha, max_iter=self.epochs,
                                    early_stopping=False, random_state=self.seed + i).fit(Z, T)
                       for i in range(self.n_models)]
        return self

    def predict(self, X):
        Z = self.sx.transform(X)
        return self.my + self.sy * np.mean([m.predict(Z) for m in self.models], axis=0)


def forecast_mlp(y, origin, h_max=CV_H, window=28, seed=SEED):
    """MIMO strategy: one MLP gives the 14 forecasts at once."""
    yy = y.loc[:origin]
    ts = yy.index[window + 1:len(yy) - h_max]
    X = np.array([mimo_inputs(yy, t, window, h_max) for t in ts])
    Y = np.array([yy.loc[t + pd.Timedelta(days=1):t + pd.Timedelta(days=h_max)].values for t in ts])
    m = MLPMulti(seed=seed).fit(X, Y)
    return m.predict(mimo_inputs(yy, origin, window, h_max)[None, :])[0]


def forecast_lstm(y, origin, h_max=CV_H, window=56, hidden=32, epochs=150, seed=SEED):
    """A small LSTM (PyTorch, CPU): input = the last 56 days of standardised load and weekday dummies, one layer of 32
    units, a linear layer with 14 outputs (MIMO); full-batch Adam, fixed seed."""
    import torch
    torch.manual_seed(seed)
    torch.set_num_threads(1)
    yy = y.loc[:origin]
    mu, sd = yy.mean(), yy.std()
    z = ((yy - mu) / sd).values
    dow = np.eye(7)[yy.index.dayofweek]
    seq = np.column_stack([z, dow]).astype(np.float32)
    idx = range(window, len(z) - h_max + 1)
    X = np.stack([seq[i - window:i] for i in idx])
    Y = np.stack([z[i:i + h_max] for i in idx]).astype(np.float32)

    class Net(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = torch.nn.LSTM(seq.shape[1], hidden, batch_first=True)
            self.out = torch.nn.Linear(hidden, h_max)

        def forward(self, x):
            o, _ = self.lstm(x)
            return self.out(o[:, -1])

    net = Net()
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    Xt, Yt = torch.tensor(X), torch.tensor(Y)
    losses = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = torch.mean((net(Xt) - Yt) ** 2)
        loss.backward()
        opt.step()
        losses.append(float(loss))
    with torch.no_grad():
        f = net(torch.tensor(seq[None, -window:])).numpy()[0]
    return mu + sd * f, losses


def fig_mlp(save=True):
    """Left: a multilayer perceptron with 3 inputs, 5 hidden neurons and one output; right: activation functions."""
    fig, ax = plt.subplots(1, 2, figsize=(8.91, 2.48), gridspec_kw={'width_ratios': [1.15, 1]})
    layers = [['$y_t$', '$y_{t-6}$', 'holiday'], [''] * 5, ['$\\hat y_{t+1}$']]
    xs = [0.1, 0.5, 0.9]
    pos = []
    for x, lay in zip(xs, layers):
        ys = np.linspace(0.85, 0.15, len(lay)) if len(lay) > 1 else [0.5]
        pos.append([(x, y) for y in ys])
    for a, b in zip(pos[:-1], pos[1:]):
        for p in a:
            for q in b:
                ax[0].plot([p[0], q[0]], [p[1], q[1]], color=st.Teal, lw=0.8, zorder=1)
    for i, (lay, ps) in enumerate(zip(layers, pos)):
        col = [st.MainBlue, st.Forest, st.IDAred][i]
        for (x, y), lab in zip(ps, lay):
            ax[0].add_patch(Circle((x, y), 0.045, facecolor=col, edgecolor='black', zorder=2))
            if lab:
                ax[0].text(x + (-0.07 if i == 0 else 0.07), y, lab, ha='right' if i == 0 else 'left', va='center',
                           fontsize=11, color='black')
    ax[0].text(0.5, 0.97, 'hidden layer: $h_j = g(b_j + w_j^\\top x)$', ha='center', fontsize=10.5, color='black')
    ax[0].text(0.9, 0.36, 'output:\n$\\hat y = c + v^\\top h$', ha='center', va='top', fontsize=10.5, color='black')
    ax[0].set_xlim(-0.25, 1.2)
    ax[0].set_ylim(0, 1.02)
    ax[0].axis('off')
    z = np.linspace(-4, 4, 400)
    ax[1].plot(z, 1 / (1 + np.exp(-z)), color=st.MainBlue, lw=1.8, label='logistic $1/(1+e^{-z})$')
    ax[1].plot(z, np.tanh(z), color=st.IDAred, lw=1.8, label='tanh $z$')
    ax[1].plot(z, np.maximum(z, 0), color=st.Forest, lw=1.8, label='ReLU $\\max(0, z)$')
    ax[1].set_ylim(-1.2, 2.5)
    ax[1].axhline(0, color='black', lw=0.6)
    ax[1].set_xlabel('$z$')
    ax[1].set_ylabel('$g(z)$')
    st.legend_outside_bottom(ax[1], ncol=2, y=-0.2)
    fig.tight_layout()
    st.check_no_grey(fig)
    if save:
        st.save_fig('tsa_ch9_mlp')
    else:
        plt.show()


def fig_rnn(save=True):
    """A recurrent network unrolled in time (left) and the LSTM cell state with its three gates (right)."""
    fig, ax = plt.subplots(1, 2, figsize=(8.49, 2.33), gridspec_kw={'width_ratios': [1.2, 1]})
    a = ax[0]
    a.set_xlim(0, 10.5)
    a.set_ylim(0, 6)
    a.axis('off')
    labs = ['$t-2$', '$t-1$', '$t$']
    for i, lab in enumerate(labs):
        x = 1 + 3.2 * i
        a.add_patch(Rectangle((x, 2.4), 1.6, 1.2, facecolor=st.Forest, edgecolor='black'))
        a.text(x + 0.8, 3.0, f'$h_{{{lab[1:-1]}}}$', ha='center', va='center', color='white', fontsize=12)
        a.add_patch(Rectangle((x + 0.2, 0.4), 1.2, 0.9, facecolor=st.MainBlue, edgecolor='black'))
        a.text(x + 0.8, 0.85, f'$x_{{{lab[1:-1]}}}$', ha='center', va='center', color='white', fontsize=12)
        a.add_patch(FancyArrowPatch((x + 0.8, 1.3), (x + 0.8, 2.4), arrowstyle='->', mutation_scale=12, color='black'))
        if i < 2:
            a.add_patch(FancyArrowPatch((x + 1.6, 3.0), (x + 3.2, 3.0), arrowstyle='->', mutation_scale=12,
                                        color=st.IDAred, lw=1.6))
    a.add_patch(Rectangle((7.6, 4.6), 1.6, 0.9, facecolor=st.IDAred, edgecolor='black'))
    a.text(8.4, 5.05, '$\\hat y_{t+1}$', ha='center', va='center', color='white', fontsize=12)
    a.add_patch(FancyArrowPatch((8.2, 3.6), (8.4, 4.6), arrowstyle='->', mutation_scale=12, color='black'))
    a.text(5.2, 5.6, '$h_t = \\tanh(W h_{t-1} + U x_t + b)$: the same weights at every step', ha='center',
           fontsize=10.5, color='black')
    b = ax[1]
    b.set_xlim(0, 10)
    b.set_ylim(0, 6)
    b.axis('off')
    b.add_patch(FancyArrowPatch((0.3, 4.8), (9.7, 4.8), arrowstyle='->', mutation_scale=14, color=st.IDAred, lw=2.4))
    b.text(0.3, 5.3, 'cell state $c_{t-1} \\rightarrow c_t$ (long memory)', fontsize=10.5, color='black')
    for x, g, txt in [(1.4, 'f', 'forget'), (4.3, 'i', 'input'), (7.2, 'o', 'output')]:
        b.add_patch(Circle((x + 0.6, 2.4), 0.6, facecolor=st.MainBlue, edgecolor='black'))
        b.text(x + 0.6, 2.4, f'${g}_t$', ha='center', va='center', color='white', fontsize=12)
        b.text(x + 0.6, 1.3, f'{txt} gate', ha='center', fontsize=10, color='black')
        b.add_patch(FancyArrowPatch((x + 0.6, 3.0), (x + 0.6, 4.7), arrowstyle='->', mutation_scale=12, color='black'))
    b.text(5.0, 0.3, '$c_t = f_t \\odot c_{t-1} + i_t \\odot \\tilde c_t$;   $h_t = o_t \\odot \\tanh(c_t)$',
           ha='center', fontsize=10.5, color='black')
    fig.tight_layout()
    st.check_no_grey(fig)
    if save:
        st.save_fig('tsa_ch9_rnn')
    else:
        plt.show()

## Run

In [ ]:
fig_mlp()
fig_rnn()
y = load_daily()
o = pd.Timestamp(CV_FIRST)
f, losses = forecast_lstm(y, o)
print('LSTM training loss, first and last epoch:', losses[0], losses[-1])
print(pd.DataFrame({'actual': y.loc[o + pd.Timedelta(days=1):].iloc[:CV_H].values, 'LSTM': f, 'MLP': forecast_mlp(y, o)}).round(3))

![tsa_ch9_mlp](./tsa_ch9_mlp.png)

Output: `tsa_ch9_mlp.pdf`

![tsa_ch9_rnn](./tsa_ch9_rnn.png)

Output: `tsa_ch9_rnn.pdf`